In [4]:
# ============================================================================
# DAY 25: JOIN OPERATIONS - COMPLETE PROGRAM
# ============================================================================
# Objective: Master JOIN operations to combine data from multiple tables
# Table Structure:
#   campaigns: campaign_name, city, format, spend, impressions, clicks, conversions, revenue
#   campaign_performance: performance_id, campaign_id, impressions, clicks, conversions, roi_percentage
#   city_demographics: city_id, city_name, population, avg_household_income
# ============================================================================

import sqlite3
# Import SQLite library for database operations

import pandas as pd
# Import pandas for data handling and DataFrames

import os
# Import os for file operations

# ============================================================================
# CELL 1: Setup & Connect to Database
# ============================================================================

print("=" * 80)
print("DAY 25: JOIN OPERATIONS - STARTING")
print("=" * 80)

# Set working directory
os.chdir(r'D:\Mission_Blitzkreig\Month_2_SQL')
# Navigate to the Month 2 SQL folder

# Connect to database
conn = sqlite3.connect('marketing_campaigns.db')
# Open connection to existing database file

# Create cursor
cursor = conn.cursor()
# Create cursor object for executing SQL commands

print("\n✅ Connected to database!")
print("✅ Ready for Day 25: JOIN Operations!")


# ============================================================================
# CELL 2: Create Secondary Table - campaign_performance
# ============================================================================

print("\n" + "=" * 80)
print("CELL 2: Creating campaign_performance table")
print("=" * 80)

# Drop old table if exists (fresh start)
cursor.execute("DROP TABLE IF EXISTS campaign_performance")
# This removes any old data to start completely fresh

# Create new campaign_performance table
cursor.execute("""
CREATE TABLE campaign_performance(
    performance_id INTEGER PRIMARY KEY,
    campaign_id INTEGER,
    impressions INTEGER,
    clicks INTEGER,
    conversions INTEGER,
    roi_percentage REAL
)
""")
# This creates a separate table for performance metrics
# campaign_id links back to campaigns table (uses rowid)

# Commit the table creation
conn.commit()
# Save the CREATE TABLE command

print("✅ campaign_performance table created successfully!")


# ============================================================================
# CELL 3: Insert Sample Data into campaign_performance
# ============================================================================

print("\n" + "=" * 80)
print("CELL 3: Inserting performance data")
print("=" * 80)

# Create sample performance data as DataFrame
sample_performance = pd.DataFrame({
    'performance_id': [1, 2, 3, 4, 5, 6],
    # Unique ID for each performance record
    
    'campaign_id': [1, 2, 3, 4, 5, 6],
    # Links to campaign rowid (relationship key)
    
    'impressions': [50000, 75000, 30000, 120000, 45000, 25000],
    # How many people saw the ad
    
    'clicks': [2500, 3456, 2456, 9000, 2311, 876],
    # How many people clicked
    
    'conversions': [250, 375, 150, 600, 225, 125],
    # How many conversions (sales/signups)
    
    'roi_percentage': [45.5, 52.4, 11.4, 67.5, 34.5, 22.4]
    # Return on investment percentage
})

# Insert data into the table
sample_performance.to_sql('campaign_performance', conn, if_exists='append', index=False)
# Use if_exists='append' to add to table

# Commit the insert
conn.commit()
# Save the data insertion

print(f"✅ Inserted {len(sample_performance)} performance records!")


# ============================================================================
# CELL 4: Verify Both Tables Exist and Show Sample Data
# ============================================================================

print("\n" + "=" * 80)
print("CELL 4: Verifying table structures")
print("=" * 80)

# Get sample from campaigns table
print("\n📊 CAMPAIGNS TABLE (First 5 rows):")
df_campaigns = pd.read_sql_query("SELECT * FROM campaigns LIMIT 5", conn)
# Execute SELECT query and convert to DataFrame

print(df_campaigns.to_string(index=False))
# Print without index numbers

# Count total campaigns
total_campaigns = len(pd.read_sql_query("SELECT * FROM campaigns", conn))
# Get total count

print(f"\nTotal campaigns in database: {total_campaigns}")
# Print total

# Get sample from campaign_performance table
print("\n📊 CAMPAIGN_PERFORMANCE TABLE (First 5 rows):")
df_perf = pd.read_sql_query("SELECT * FROM campaign_performance", conn)
# Execute SELECT query

print(df_perf.to_string(index=False))
# Print without index numbers

print(f"\nTotal performance records: {len(df_perf)}")
# Print total


# ============================================================================
# CELL 5: INNER JOIN - Only Matching Records from Both Tables
# ============================================================================

print("\n" + "=" * 80)
print("CELL 5: INNER JOIN - Campaigns with Performance")
print("=" * 80)

query = """
SELECT 
    c.rowid as campaign_id,
    c.campaign_name,
    c.city,
    c.format,
    c.spend,
    c.revenue,
    p.impressions,
    p.clicks,
    p.conversions,
    p.roi_percentage
FROM campaigns c
INNER JOIN campaign_performance p ON c.rowid = p.campaign_id
ORDER BY c.campaign_name
"""
# INNER JOIN only returns rows where campaign_id exists in BOTH tables
# c.rowid is the implicit primary key in campaigns table
# ORDER BY campaign_name sorts results alphabetically

df_inner = pd.read_sql_query(query, conn)
# Execute query and convert to DataFrame

print("\n🔗 INNER JOIN RESULT:")
print(df_inner.to_string(index=False))
# Print all columns

print(f"\nRows returned: {len(df_inner)}")
# Count of matched records - should match number of performance records


# ============================================================================
# CELL 6: LEFT JOIN - All from Left Table (campaigns)
# ============================================================================

print("\n" + "=" * 80)
print("CELL 6: LEFT JOIN - All Campaigns with Performance Data")
print("=" * 80)

query = """
SELECT 
    c.rowid as campaign_id,
    c.campaign_name,
    c.city,
    c.format,
    c.spend,
    c.revenue,
    COALESCE(p.impressions, 0) as impressions,
    COALESCE(p.clicks, 0) as clicks,
    COALESCE(p.conversions, 0) as conversions,
    COALESCE(p.roi_percentage, 0) as roi_percentage
FROM campaigns c
LEFT JOIN campaign_performance p ON c.rowid = p.campaign_id
ORDER BY c.campaign_name
"""
# LEFT JOIN keeps ALL rows from left table (campaigns)
# Adds performance data if it exists, fills with 0 if NULL
# COALESCE replaces NULL values with 0

df_left = pd.read_sql_query(query, conn)
# Execute query

print("\n⬅️ LEFT JOIN RESULT:")
print(df_left.to_string(index=False))
# Print all campaigns with performance or zeros

# Count campaigns without performance data
no_perf = len(df_left[df_left['impressions'] == 0])
# Find rows where impressions is 0 (no performance data)

print(f"\nTotal campaigns: {len(df_left)}")
# Total number of campaigns

print(f"Campaigns WITHOUT performance data: {no_perf}")
# Count campaigns without matching performance records


# ============================================================================
# CELL 7: RIGHT JOIN - All from Right Table (campaign_performance)
# ============================================================================

print("\n" + "=" * 80)
print("CELL 7: RIGHT JOIN - All Performance Records")
print("=" * 80)

query = """
SELECT 
    p.performance_id,
    p.campaign_id,
    COALESCE(c.campaign_name, 'Unknown Campaign') as campaign_name,
    COALESCE(c.city, 'Unknown') as city,
    COALESCE(c.format, 'Unknown') as format,
    p.impressions,
    p.clicks,
    p.conversions,
    p.roi_percentage
FROM campaigns c
RIGHT JOIN campaign_performance p ON c.rowid = p.campaign_id
ORDER BY p.roi_percentage DESC
"""
# RIGHT JOIN keeps ALL rows from right table (campaign_performance)
# Adds campaign info if it exists
# COALESCE handles missing campaign data

df_right = pd.read_sql_query(query, conn)
# Execute query

print("\n➡️ RIGHT JOIN RESULT:")
print(df_right.to_string(index=False))
# Print all performance records

print(f"\nTotal performance records: {len(df_right)}")
# Count of performance records


# ============================================================================
# CELL 8: JOIN with WHERE Clause - Filter Joined Results
# ============================================================================

print("\n" + "=" * 80)
print("CELL 8: JOIN with WHERE - High-ROI Mumbai Campaigns")
print("=" * 80)

query = """
SELECT 
    c.rowid as campaign_id,
    c.campaign_name,
    c.city,
    c.format,
    c.spend,
    c.revenue,
    p.impressions,
    p.clicks,
    p.conversions,
    p.roi_percentage
FROM campaigns c
INNER JOIN campaign_performance p ON c.rowid = p.campaign_id
WHERE c.city = 'Mumbai' AND p.roi_percentage > 40
ORDER BY p.roi_percentage DESC
"""
# INNER JOIN campaigns with performance
# WHERE filters to only Mumbai campaigns with ROI > 40%
# WHERE conditions filter AFTER joining

df_filtered = pd.read_sql_query(query, conn)
# Execute query

print("\n🎯 FILTERED JOIN RESULT:")
print(df_filtered.to_string(index=False))
# Print filtered results

if len(df_filtered) > 0:
    # Check if any results exist
    
    avg_roi = df_filtered['roi_percentage'].mean()
    # Calculate average ROI of filtered campaigns
    
    print(f"\n✅ Found {len(df_filtered)} Mumbai campaigns with ROI > 40%")
    # Count matching campaigns
    
    print(f"Average ROI: {avg_roi:.2f}%")
    # Show average ROI
else:
    # No matching records
    
    print("\n❌ No Mumbai campaigns with ROI > 40%")
    # Message if no results


# ============================================================================
# CELL 9: JOIN with GROUP BY & Aggregates - Aggregate by City
# ============================================================================

print("\n" + "=" * 80)
print("CELL 9: JOIN with GROUP BY - City Performance Metrics")
print("=" * 80)

query = """
SELECT 
    c.city,
    COUNT(c.rowid) as total_campaigns,
    SUM(c.spend) as total_spend,
    SUM(c.revenue) as total_revenue,
    ROUND((SUM(c.revenue) - SUM(c.spend)) * 100.0 / SUM(c.spend), 2) as roi_percentage,
    ROUND(AVG(c.spend), 2) as avg_spend_per_campaign,
    ROUND(AVG(c.revenue), 2) as avg_revenue_per_campaign,
    ROUND(AVG(p.impressions), 0) as avg_impressions,
    ROUND(AVG(p.clicks), 0) as avg_clicks,
    ROUND(AVG(p.roi_percentage), 2) as avg_performance_roi
FROM campaigns c
LEFT JOIN campaign_performance p ON c.rowid = p.campaign_id
GROUP BY c.city
ORDER BY total_revenue DESC
"""
# LEFT JOIN includes all campaigns
# GROUP BY city aggregates metrics by city
# ROUND creates clean numbers
# SUM = total, AVG = average, COUNT = count

df_city_metrics = pd.read_sql_query(query, conn)
# Execute query

print("\n📊 CITY PERFORMANCE METRICS:")
print(df_city_metrics.to_string(index=False))
# Print city aggregates

if len(df_city_metrics) > 0:
    # Check if results exist
    
    top_city = df_city_metrics.iloc[0]
    # Get first row (highest revenue city)
    
    print(f"\n🏆 TOP CITY BY REVENUE:")
    print(f"   City: {top_city['city']}")
    # City name
    
    print(f"   Total Spend: ₹{top_city['total_spend']:,.0f}")
    # Total budget spent
    
    print(f"   Total Revenue: ₹{top_city['total_revenue']:,.0f}")
    # Total revenue generated
    
    print(f"   Campaign ROI: {top_city['roi_percentage']:.2f}%")
    # Overall ROI for city
    
    print(f"   Total Campaigns: {int(top_city['total_campaigns'])}")
    # Number of campaigns
    
    print(f"   Avg Spend per Campaign: ₹{top_city['avg_spend_per_campaign']:,.0f}")
    # Average budget per campaign
    
    print(f"   Avg Revenue per Campaign: ₹{top_city['avg_revenue_per_campaign']:,.0f}")
    # Average revenue per campaign
    
    print(f"   Avg Performance ROI: {top_city['avg_performance_roi']:.2f}%")
    # Average ROI from performance table


# ============================================================================
# CELL 10: Create Third Table - city_demographics
# ============================================================================

print("\n" + "=" * 80)
print("CELL 10: Creating city_demographics table")
print("=" * 80)

# Drop old table if exists
cursor.execute("DROP TABLE IF EXISTS city_demographics")
# Remove any existing city_demographics table

# Create new table
cursor.execute("""
CREATE TABLE city_demographics(
    city_id INTEGER PRIMARY KEY,
    city_name TEXT,
    population INTEGER,
    avg_household_income REAL
)
""")
# Create table with city demographic information

# Create sample data
demo_data = pd.DataFrame({
    'city_id': [1, 2, 3, 4, 5, 6],
    # Unique ID for each city
    
    'city_name': ['Mumbai', 'Chennai', 'Delhi', 'Bangalore', 'Goa', 'Pune'],
    # City names (must match campaigns table cities)
    
    'population': [20000000, 7000000, 16000000, 8500000, 1500000, 6000000],
    # Population of each city
    
    'avg_household_income': [45000, 35000, 50000, 48000, 52000, 40000]
    # Average household income
})

# Insert data into table
demo_data.to_sql('city_demographics', conn, if_exists='append', index=False)
# Add demographic data to table

# Commit the insert
conn.commit()
# Save data

print("✅ city_demographics table created with sample data!")
print(f"✅ Inserted {len(demo_data)} cities")


# ============================================================================
# CELL 11: Triple JOIN - Combine All Three Tables
# ============================================================================

print("\n" + "=" * 80)
print("CELL 11: Triple JOIN - All Three Tables Combined")
print("=" * 80)

query = """
SELECT 
    c.rowid as campaign_id,
    c.campaign_name,
    c.city,
    c.format,
    c.spend,
    c.revenue,
    p.impressions,
    p.clicks,
    p.conversions,
    p.roi_percentage,
    d.population,
    d.avg_household_income,
    ROUND(CAST(c.revenue AS REAL) / d.population * 1000000, 2) as revenue_per_million
FROM campaigns c
LEFT JOIN campaign_performance p ON c.rowid = p.campaign_id
LEFT JOIN city_demographics d ON c.city = d.city_name
ORDER BY revenue_per_million DESC
"""
# Triple JOIN combines all three tables
# Uses LEFT JOINs to keep all campaigns
# Calculates revenue per million people

df_triple = pd.read_sql_query(query, conn)
# Execute query

print("\n🌍 TRIPLE JOIN RESULT:")
print(df_triple.to_string(index=False))
# Print combined data

if len(df_triple) > 0:
    # Check if results exist
    
    print(f"\n💡 INSIGHTS:")
    print(f"Best revenue per million people: ₹{df_triple.iloc[0]['revenue_per_million']:,.2f}")
    # Highest revenue efficiency metric
    
    print(f"Campaign: {df_triple.iloc[0]['campaign_name']}")
    # Campaign name with best efficiency
    
    print(f"City: {df_triple.iloc[0]['city']}")
    # City of best campaign
    
    print(f"Population: {int(df_triple.iloc[0]['population']):,}")
    # City population


# ============================================================================
# CELL 12: Executive Summary - Complete Analytics Report
# ============================================================================

print("\n" + "=" * 80)
print("CELL 12: EXECUTIVE SUMMARY - Complete Analytics Report")
print("=" * 80)

query = """
SELECT 
    d.city_name,
    COUNT(DISTINCT c.rowid) as total_campaigns,
    SUM(c.spend) as total_spend,
    SUM(c.revenue) as total_revenue,
    ROUND((SUM(c.revenue) - SUM(c.spend)) * 100.0 / SUM(c.spend), 2) as roi_percentage,
    ROUND(AVG(c.revenue), 2) as avg_revenue_per_campaign,
    ROUND(AVG(p.roi_percentage), 2) as avg_performance_roi,
    d.population,
    ROUND(SUM(c.revenue) * 1000000.0 / d.population, 2) as revenue_per_million_people
FROM city_demographics d
LEFT JOIN campaigns c ON d.city_name = c.city
LEFT JOIN campaign_performance p ON c.rowid = p.campaign_id
GROUP BY d.city_name
ORDER BY total_revenue DESC
"""
# Complete city analysis with all metrics
# Calculates ROI, revenue per campaign, demographics
# Uses GROUP BY to aggregate by city

df_executive = pd.read_sql_query(query, conn)
# Execute query

print("\n" + "=" * 100)
print("📊 COMPLETE EXECUTIVE SUMMARY - CITY PERFORMANCE ANALYSIS")
print("=" * 100)
print(df_executive.to_string(index=False))
# Print complete report

# Show top performing city
if len(df_executive) > 0 and df_executive.iloc[0]['total_campaigns'] > 0:
    # Check if results and data exist
    
    top = df_executive.iloc[0]
    # Get best city (first row, ordered by revenue DESC)
    
    print("\n" + "=" * 100)
    print("🏆 TOP PERFORMING CITY")
    print("=" * 100)
    print(f"City: {top['city_name']}")
    # City name
    
    print(f"Total Campaigns: {int(top['total_campaigns'])}")
    # Number of campaigns
    
    print(f"Total Spend: ₹{top['total_spend']:,.0f}")
    # Total budget spent
    
    print(f"Total Revenue: ₹{top['total_revenue']:,.0f}")
    # Total revenue generated
    
    print(f"Campaign ROI: {top['roi_percentage']:.2f}%")
    # Overall return on investment
    
    print(f"Avg Revenue per Campaign: ₹{top['avg_revenue_per_campaign']:,.0f}")
    # Average revenue per campaign
    
    print(f"Avg Performance ROI: {top['avg_performance_roi']:.2f}%")
    # Average ROI from performance metrics
    
    print(f"City Population: {int(top['population']):,}")
    # Total city population
    
    print(f"Revenue per Million People: ₹{top['revenue_per_million_people']:,.0f}")
    # Efficiency metric


# ============================================================================
# CELL 13: Close Database Connection
# ============================================================================

print("\n" + "=" * 80)
print("CELL 13: Closing Database Connection")
print("=" * 80)

# Close cursor
cursor.close()
# Close the cursor object

# Close connection
conn.close()
# Close the database connection

print("\n" + "=" * 80)
print("✅ DAY 25 COMPLETE - JOIN OPERATIONS MASTERED!")
print("=" * 80)
print("\n📚 WHAT YOU LEARNED:")
print("   ✅ INNER JOIN - matching records only")
print("   ✅ LEFT JOIN - all from left table")
print("   ✅ RIGHT JOIN - all from right table")
print("   ✅ COALESCE - replace NULL with default")
print("   ✅ JOIN + WHERE - filter joined data")
print("   ✅ JOIN + GROUP BY - aggregate joined data")
print("   ✅ TRIPLE JOIN - combine 3 tables")
print("   ✅ Executive reporting with JOINs")
print("\n🚀 NEXT: Day 26 - Subqueries & Advanced SQL")
print("=" * 80)


DAY 25: JOIN OPERATIONS - STARTING

✅ Connected to database!
✅ Ready for Day 25: JOIN Operations!

CELL 2: Creating campaign_performance table
✅ campaign_performance table created successfully!

CELL 3: Inserting performance data
✅ Inserted 6 performance records!

CELL 4: Verifying table structures

📊 CAMPAIGNS TABLE (First 5 rows):
             campaign_name    city      format  spend  impressions  clicks  conversions  revenue
  Mumbai Centre - 48 Sheet  Mumbai    48 Sheet   4200       980000    1240           87    18900
  Delhi Motorway - Digital   Delhi     Digital   2800       620000     890           54    11200
Chennai Roadside - 6 Sheet Chennai     6 Sheet   1500       310000     420           28     5600
Mumbai Airport - Supersite  Mumbai   Supersite   8500      1850000    3100          210    48000
   Pune City - Bus Shelter    Pune Bus Shelter    950       180000     210           15     2800

Total campaigns in database: 10

📊 CAMPAIGN_PERFORMANCE TABLE (First 5 rows):
 per